In [4]:
import os
from pathlib import Path
from urllib.parse import quote

import requests
from tqdm.auto import tqdm
from tqdm.utils import CallbackIOWrapper

TOKEN = "DUrFDrr7KuiNJhkTisR14dGg06iD6xGreDk3YZKW3yPoD0MzY2eFdXkq6aE1"   # or paste your token directly
TOKEN

'DUrFDrr7KuiNJhkTisR14dGg06iD6xGreDk3YZKW3yPoD0MzY2eFdXkq6aE1'

In [7]:
BASE_URL = "https://zenodo.org"
DEPOSITION_ID = 18703669
FILE = Path("../datasets/imagenet_v5_rot_10m_subdataset/imagenet_v5_rot_10m_1mSplits_00.h5")

headers = {"Authorization": f"Bearer {TOKEN}"}

# 1. Get the existing draft
r = requests.get(
    f"{BASE_URL}/api/deposit/depositions/{DEPOSITION_ID}",
    headers=headers,
    timeout=60,
)
r.raise_for_status()
dep = r.json()

bucket_url = dep["links"]["bucket"]

print(dep["id"], dep["state"], dep["submitted"])
print(dep["links"]["bucket"])

18703669 unsubmitted False
https://zenodo.org/api/files/fe26f58e-22da-4111-bfe9-63ade0dedf0a


In [ ]:
bucket_url = dep["links"]["bucket"]
file_path = Path("../output.png")

with file_path.open("rb") as f:
    r = requests.put(
        f"{bucket_url}/{quote(file_path.name)}",
        data=f,
        headers={"Authorization": f"Bearer {TOKEN}"},
        timeout=600,
    )

print(r.status_code)
print(r.text[:500])

201
{"created": "2026-03-13T17:51:28.862599+00:00", "updated": "2026-03-13T17:51:29.127179+00:00", "version_id": "d351b380-1a93-4a8c-9a9c-d0830ad50a56", "key": "output.png", "size": 305861, "mimetype": "image/png", "checksum": "md5:31f7b347585a98fb07eabb1cbe6ff986", "is_head": true, "delete_marker": false, "links": {"self": "https://zenodo.org/api/files/fe26f58e-22da-4111-bfe9-63ade0dedf0a/output.png", "version": "https://zenodo.org/api/files/fe26f58e-22da-4111-bfe9-63ade0dedf0a/output.png?version_i


In [8]:
# 2. Upload the file with a progress bar
upload_url = f"{bucket_url}/{quote(FILE.name)}"
size = FILE.stat().st_size

upload_headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/octet-stream",
    "Content-Length": str(size),
}

with FILE.open("rb") as f, tqdm(
    total=size,
    unit="B",
    unit_scale=True,
    unit_divisor=1024,
    desc=FILE.name,
) as bar:
    wrapped = CallbackIOWrapper(bar.update, f, "read")
    r = requests.put(upload_url, data=wrapped, headers=upload_headers, timeout=3600)
    r.raise_for_status()

print("Upload complete")
print(r.json())

imagenet_v5_rot_10m_1mSplits_00.h5:  76%|███████▌  | 130G/171G [12:31:01<3:56:35, 3.09MB/s] 


SSLError: HTTPSConnectionPool(host='zenodo.org', port=443): Max retries exceeded with url: /api/files/fe26f58e-22da-4111-bfe9-63ade0dedf0a/imagenet_v5_rot_10m_1mSplits_00.h5 (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:2427)')))

In [ ]:
import requests
from tqdm import tqdm
import os

TOKEN = "DUrFDrr7KuiNJhkTisR14dGg06iD6xGreDk3YZKW3yPoD0MzY2eFdXkq6aE1"
BUCKET_URL = "https://zenodo.org/api/files/fe26f58e-22da-4111-bfe9-63ade0dedf0a"

FILE = "../datasets/imagenet_v5_rot_10m_subdataset/imagenet_v5_rot_10m_1mSplits_00.h5"

filesize = os.path.getsize(FILE)

with open(FILE, "rb") as f:
    with tqdm(total=filesize, unit="B", unit_scale=True) as pbar:
        def gen():
            while True:
                chunk = f.read(1024*1024)
                if not chunk:
                    break
                pbar.update(len(chunk))
                yield chunk

        r = requests.put(
            f"{BUCKET_URL}/{FILE}",
            data=gen(),
            params={"access_token": TOKEN}
        )

print(r.status_code)


  0%|          | 60.8M/183G [00:54<45:25:10, 1.12MB/s]


KeyboardInterrupt: 

In [1]:
import os
import time
from pathlib import Path
import requests
from tqdm.auto import tqdm

BUCKET_URL = "https://zenodo.org/api/files/fe26f58e-22da-4111-bfe9-63ade0dedf0a"
TOKEN = "DUrFDrr7KuiNJhkTisR14dGg06iD6xGreDk3YZKW3yPoD0MzY2eFdXkq6aE1"
PART_DIR = Path("../datasets/imagenet_v5_rot_10m_subdataset")
FILES = sorted(PART_DIR.glob("imagenet_v5_rot_10m_1mSplits_01.part_*"))  # or "imagenet_v5_rot_10m_1mSplits_01.part_*"

MAX_TRIES = 8

for FILE in FILES:
    print(f"Uploading {FILE.name} ...")
    filesize = os.path.getsize(FILE)

    ok = False
    for attempt in range(1, MAX_TRIES + 1):
        try:
            with open(FILE, "rb") as f:
                with tqdm(total=filesize, unit="B", unit_scale=True, desc=FILE.name) as pbar:
                    def gen():
                        while True:
                            chunk = f.read(1024 * 1024)  # same 1MB chunk style
                            if not chunk:
                                break
                            pbar.update(len(chunk))
                            yield chunk

                    r = requests.put(
                        f"{BUCKET_URL.rstrip('/')}/{FILE.name}",  # important fix
                        data=gen(),
                        params={"access_token": TOKEN},
                        timeout=(30, 3600),
                    )

            print(r.status_code)
            r.raise_for_status()
            ok = True
            break

        except Exception as e:
            print(f"Attempt {attempt}/{MAX_TRIES} failed for {FILE.name}: {e}")
            if attempt < MAX_TRIES:
                time.sleep(min(120, 2 ** attempt))
            else:
                raise

    if not ok:
        raise RuntimeError(f"Failed to upload {FILE.name}")


/mnt/scratch/home/yichen/anaconda3/envs/symmetry/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Uploading imagenet_v5_rot_10m_1mSplits_01.part_000 ...


imagenet_v5_rot_10m_1mSplits_01.part_000:   0%|          | 0.00/21.5G [00:00<?, ?B/s]

imagenet_v5_rot_10m_1mSplits_01.part_000: 100%|██████████| 21.5G/21.5G [1:21:26<00:00, 4.40MB/s]


400
Attempt 1/8 failed for imagenet_v5_rot_10m_1mSplits_01.part_000: 400 Client Error: BAD REQUEST for url: https://zenodo.org/api/files/fe26f58e-22da-4111-bfe9-63ade0dedf0a/imagenet_v5_rot_10m_1mSplits_01.part_000?access_token=DUrFDrr7KuiNJhkTisR14dGg06iD6xGreDk3YZKW3yPoD0MzY2eFdXkq6aE1


...

imagenet_v5_rot_10m_1mSplits_01.part_000: 100%|██████████| 21.5G/21.5G [46:40<00:00, 7.67MB/s]  


400
Attempt 7/8 failed for imagenet_v5_rot_10m_1mSplits_01.part_000: 400 Client Error: BAD REQUEST for url: https://zenodo.org/api/files/fe26f58e-22da-4111-bfe9-63ade0dedf0a/imagenet_v5_rot_10m_1mSplits_01.part_000?access_token=DUrFDrr7KuiNJhkTisR14dGg06iD6xGreDk3YZKW3yPoD0MzY2eFdXkq6aE1


imagenet_v5_rot_10m_1mSplits_01.part_000: 100%|██████████| 21.5G/21.5G [45:41<00:00, 7.83MB/s]  


400
Attempt 8/8 failed for imagenet_v5_rot_10m_1mSplits_01.part_000: 400 Client Error: BAD REQUEST for url: https://zenodo.org/api/files/fe26f58e-22da-4111-bfe9-63ade0dedf0a/imagenet_v5_rot_10m_1mSplits_01.part_000?access_token=DUrFDrr7KuiNJhkTisR14dGg06iD6xGreDk3YZKW3yPoD0MzY2eFdXkq6aE1


HTTPError: 400 Client Error: BAD REQUEST for url: https://zenodo.org/api/files/fe26f58e-22da-4111-bfe9-63ade0dedf0a/imagenet_v5_rot_10m_1mSplits_01.part_000?access_token=DUrFDrr7KuiNJhkTisR14dGg06iD6xGreDk3YZKW3yPoD0MzY2eFdXkq6aE1

In [1]:
import os
import time
import random
import requests
from pathlib import Path
from tqdm.auto import tqdm

BUCKET_URL = "https://zenodo.org/api/files/fe26f58e-22da-4111-bfe9-63ade0dedf0a"
FILE = Path("../datasets/imagenet_v5_rot_10m_subdataset/imagenet_v5_rot_10m_1mSplits_01.h5")
TOKEN = 'DUrFDrr7KuiNJhkTisR14dGg06iD6xGreDk3YZKW3yPoD0MzY2eFdXkq6aE1'  # put token in env, do not hardcode

MAX_TRIES = 8
CHUNK_SIZE = 16 * 1024 * 1024  # 16 MB
TIMEOUT = (30, 3600)  # connect, read

def upload_once():
    size = FILE.stat().st_size
    url = f"{BUCKET_URL.rstrip('/')}/{FILE.name}"

    with open(FILE, "rb") as f, tqdm(total=size, unit="B", unit_scale=True, desc=FILE.name) as pbar:
        def gen():
            while True:
                chunk = f.read(CHUNK_SIZE)
                if not chunk:
                    break
                pbar.update(len(chunk))
                yield chunk

        r = requests.put(
            url,
            data=gen(),
            params={"access_token": TOKEN},
            headers={
                "Content-Length": str(size),
                "Content-Type": "application/octet-stream",
            },
            timeout=TIMEOUT,
        )
    r.raise_for_status()
    return r

for attempt in range(1, MAX_TRIES + 1):
    try:
        resp = upload_once()
        print("Upload success:", resp.status_code)
        print(resp.text[:300])
        break
    except (requests.exceptions.SSLError,
            requests.exceptions.ConnectionError,
            requests.exceptions.Timeout,
            requests.exceptions.ChunkedEncodingError) as e:
        if attempt == MAX_TRIES:
            raise
        wait = min(120, 2 ** attempt) + random.uniform(0, 2)
        print(f"Attempt {attempt}/{MAX_TRIES} failed: {e}. Retrying in {wait:.1f}s...")
        time.sleep(wait)


/mnt/scratch/home/yichen/anaconda3/envs/symmetry/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
imagenet_v5_rot_10m_1mSplits_01.h5:   0%|          | 16.8M/183G [00:00<1:26:54, 35.1MB/s]


Attempt 1/8 failed: HTTPSConnectionPool(host='zenodo.org', port=443): Max retries exceeded with url: /api/files/fe26f58e-22da-4111-bfe9-63ade0dedf0a/imagenet_v5_rot_10m_1mSplits_01.h5?access_token=DUrFDrr7KuiNJhkTisR14dGg06iD6xGreDk3YZKW3yPoD0MzY2eFdXkq6aE1 (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:2427)'))). Retrying in 2.7s...


imagenet_v5_rot_10m_1mSplits_01.h5:   0%|          | 16.8M/183G [00:00<1:33:07, 32.8MB/s]


Attempt 2/8 failed: HTTPSConnectionPool(host='zenodo.org', port=443): Max retries exceeded with url: /api/files/fe26f58e-22da-4111-bfe9-63ade0dedf0a/imagenet_v5_rot_10m_1mSplits_01.h5?access_token=DUrFDrr7KuiNJhkTisR14dGg06iD6xGreDk3YZKW3yPoD0MzY2eFdXkq6aE1 (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:2427)'))). Retrying in 4.3s...


imagenet_v5_rot_10m_1mSplits_01.h5:   0%|          | 16.8M/183G [00:00<1:44:39, 29.2MB/s]


...

imagenet_v5_rot_10m_1mSplits_01.h5:   0%|          | 16.8M/183G [00:00<1:25:01, 35.9MB/s]


Attempt 5/8 failed: HTTPSConnectionPool(host='zenodo.org', port=443): Max retries exceeded with url: /api/files/fe26f58e-22da-4111-bfe9-63ade0dedf0a/imagenet_v5_rot_10m_1mSplits_01.h5?access_token=DUrFDrr7KuiNJhkTisR14dGg06iD6xGreDk3YZKW3yPoD0MzY2eFdXkq6aE1 (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:2427)'))). Retrying in 33.7s...


imagenet_v5_rot_10m_1mSplits_01.h5:   0%|          | 16.8M/183G [00:00<1:26:38, 35.2MB/s]


Attempt 6/8 failed: HTTPSConnectionPool(host='zenodo.org', port=443): Max retries exceeded with url: /api/files/fe26f58e-22da-4111-bfe9-63ade0dedf0a/imagenet_v5_rot_10m_1mSplits_01.h5?access_token=DUrFDrr7KuiNJhkTisR14dGg06iD6xGreDk3YZKW3yPoD0MzY2eFdXkq6aE1 (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:2427)'))). Retrying in 66.0s...


KeyboardInterrupt: 

In [ ]:
FILE="imagenet_v5_rot_10m_1mSplits_01.h5"
BUCKET_URL="https://zenodo.org/api/files/fe26f58e-22da-4111-bfe9-63ade0dedf0a"
TOKEN="x3PeZbLpmkufnS5Ldhi7HaveCrOUGGTcHPXVsI6GXWRCoA0wjxBwmmnSDRTx"

curl --http1.1 --tlsv1.2 --retry 8 --retry-all-errors \
  --progress-bar -X PUT --upload-file "$FILE" \
  "${BUCKET_URL%/}/$(basename "$FILE")?access_token=$TOKEN"


In [7]:
import ssl, requests, urllib3, certifi
print("OpenSSL:", ssl.OPENSSL_VERSION)
print("requests:", requests.__version__)
print("urllib3:", urllib3.__version__)
print("certifi:", certifi.where())
print(requests.get("https://zenodo.org/api/records", timeout=20).status_code)

OpenSSL: OpenSSL 3.0.16 11 Feb 2025
requests: 2.32.3
urllib3: 2.3.0
certifi: /mnt/scratch/home/yichen/anaconda3/envs/symmetry/lib/python3.11/site-packages/certifi/cacert.pem
200


In [5]:
import os
from pathlib import Path
import requests
from tqdm import tqdm

FILE = Path(FILE)
url = f"{BUCKET_URL.rstrip('/')}/{FILE.name}"
size = FILE.stat().st_size
chunk_size = 16 * 1024 * 1024  # 16 MB

def stream_file(path, pbar):
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            pbar.update(len(chunk))
            yield chunk

with tqdm(total=size, unit="B", unit_scale=True) as pbar:
    r = requests.put(
        url,
        data=stream_file(FILE, pbar),
        params={"access_token": TOKEN},
        headers={"Content-Length": str(size)},
        timeout=(30, 3600),
    )

print(r.status_code, r.text[:300])
r.raise_for_status()


  0%|          | 16.8M/183G [00:00<1:12:08, 42.3MB/s]


SSLError: HTTPSConnectionPool(host='zenodo.org', port=443): Max retries exceeded with url: /api/files/fe26f58e-22da-4111-bfe9-63ade0dedf0a/imagenet_v5_rot_10m_1mSplits_01.h5?access_token=x3PeZbLpmkufnS5Ldhi7HaveCrOUGGTcHPXVsI6GXWRCoA0wjxBwmmnSDRTx (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:2427)')))

In [9]:
import ssl
from pathlib import Path
import requests
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from tqdm.utils import CallbackIOWrapper

class TLS12Adapter(HTTPAdapter):
    def init_poolmanager(self, *args, **kwargs):
        ctx = ssl.create_default_context()
        ctx.minimum_version = ssl.TLSVersion.TLSv1_2
        kwargs["ssl_context"] = ctx
        return super().init_poolmanager(*args, **kwargs)

p = Path(FILE)
url = f"{BUCKET_URL.rstrip('/')}/{p.name}"
size = p.stat().st_size

s = requests.Session()
s.mount("https://", TLS12Adapter())
s.trust_env = False  # ignore HTTPS_PROXY/HTTP_PROXY from env

with open(p, "rb") as f, tqdm(total=size, unit="B", unit_scale=True, desc=p.name) as pbar:
    wrapped = CallbackIOWrapper(pbar.update, f, "read")
    r = s.put(
        url,
        data=wrapped,
        params={"access_token": TOKEN},
        headers={"Content-Length": str(size)},
        timeout=(30, 3600),
    )

print(r.status_code, r.text[:300])
r.raise_for_status()


/mnt/scratch/home/yichen/anaconda3/envs/symmetry/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
imagenet_v5_rot_10m_1mSplits_01.h5:   0%|          | 250M/183G [07:46<94:53:34, 536kB/s]   


KeyboardInterrupt: 

In [1]:
%load_ext autoreload
%autoreload 2
import re
import h5py
from datafed.CommandLib import API
df_api = API()

from dl_utils.utils.datafed_functions import datafed_download, datafed_upload, datafed_create_collection, datafed_upload_folder

In [4]:
datafed_download(file_path='../../', file_id='d/525673422', wait=False)

(item {
  id: "d/525673422"
  title: "test_phantom_plume.h5"
  owner: "p/2023_pld_plume"
  size: 7811176448
}
task {
  id: "task/525863775"
  type: TT_DATA_GET
  status: TS_READY
  client: "u/yig319"
  step: 0
  steps: 2
  msg: "Pending"
  ct: 1747255687
  ut: 1747255687
  source: "d/525673422"
  dest: "46c32934-b552-11ee-b08b-4bb870e392e2/scratch/home/yichen"
}
, 'DataGetReply')


In [3]:
datafed_upload('../datasets/imagenet_v5_rot_10m_fix_vector-hierarchy_metadata.h5', 'c/525611223', wait=False)

(item {
  id: "d/525681224"
  title: "imagenet_v5_rot_10m_fix_vector-hierarchy_metadata.h5"
  size: 0
  owner: "p/2025-symmetry-datasets"
}
task {
  id: "task/525870887"
  type: TT_DATA_PUT
  status: TS_READY
  client: "u/yig319"
  step: 0
  steps: 2
  msg: "Pending"
  ct: 1759285456
  ut: 1759285456
  source: "46c32934-b552-11ee-b08b-4bb870e392e2/scratch/home/yichen/Understanding-Experimental-Images-by-Identifying-Symmetries-with-Deep-Learning/datasets/imagenet_v5_rot_10m_fix_vector-hierarchy_metadata.h5"
  dest: "d/525681224"
}
, 'DataPutReply')


In [9]:
def latest_n_files(files, folder_path, pattern=r'epoch_(\d+)', n=10):
    """
    Selects the latest N files based on epoch number extracted from filename.

    Parameters:
    - files (list): List of filenames.
    - folder_path (str): Path to the folder (unused, but keeps interface consistent).
    - pattern (str): Regex pattern with a capturing group for the epoch number.
    - n (int): Number of latest files to return.

    Returns:
    - list: Selected filenames sorted by descending epoch number.
    """
    matched_files = []
    for file in files:
        match = re.search(pattern, file)
        if match:
            epoch_num = int(match.group(1))
            matched_files.append((epoch_num, file))
    matched_files.sort(reverse=True)
    selected_files = [file for _, file in matched_files[:n]]
    return selected_files

In [ ]:
# Start the recursive upload with the rule
root_folder = "../models/"
datafed_upload_folder(
    root_folder,
    parent_id='c/525611224',
    rule=lambda files, path: latest_n_files(files, path, n=10), 
    wait=False,
)

Created collection 'models' with ID: c/525611225
Created collection 'FPN' with ID: c/525611226
Created collection '09232024-FPN-dataset_v5_size-10k' with ID: c/525611227
Uploading file 'model_epoch_3000.pth' to collection '09232024-FPN-dataset_v5_size-10k'...
(item {
  id: "d/525680874"
  title: "model_epoch_3000.pth"
  size: 0
  owner: "p/2025-symmetry-datasets"
}
task {
  id: "task/525870483"
  type: TT_DATA_PUT
  status: TS_READY
  client: "u/yig319"
  step: 0
  steps: 2
  msg: "Pending"
  ct: 1757707822
  ut: 1757707822
  source: "46c32934-b552-11ee-b08b-4bb870e392e2/scratch/home/yichen/Understanding-Experimental-Images-by-Identifying-Symmetries-with-Deep-Learning/models/FPN/09232024-FPN-dataset_v5_size-10k/model_epoch_3000.pth"
  dest: "d/525680874"
}
, 'DataPutReply')
Uploaded 'model_epoch_3000.pth'
Uploading file 'model_epoch_2970.pth' to collection '09232024-FPN-dataset_v5_size-10k'...
(item {
  id: "d/525680875"
  title: "model_epoch_2970.pth"
  size: 0
  owner: "p/2025-symmet

In [2]:
# Example usage
folder_path = "../models/FPN"
parent_id = 'c/503917703'  # Optionally set a parent collection ID

datafed_upload_folder(folder_path, parent_id, metadata=None, wait=False)

Created collection 'FPN' with ID: c/525610931
Creating sub-collection for folder '09232024-FPN-dataset_v5_size-10k' within 'FPN'...
Created collection '09232024-FPN-dataset_v5_size-10k' with ID: c/525610932
Uploading file 'model_epoch_1530.pth' to collection '09232024-FPN-dataset_v5_size-10k'...
(item {
  id: "d/525662143"
  title: "model_epoch_1530.pth"
  size: 0
  owner: "p/2023_wallpaper_group_symmetry"
}
task {
  id: "task/525801675"
  type: TT_DATA_PUT
  status: TS_READY
  client: "u/yig319"
  step: 0
  steps: 2
  msg: "Pending"
  ct: 1741665850
  ut: 1741665850
  source: "46c32934-b552-11ee-b08b-4bb870e392e2/mnt/scratch/home/yichen/Understanding-Experimental-Images-by-Identifying-Symmetries-with-Deep-Learning/models/FPN/09232024-FPN-dataset_v5_size-10k/model_epoch_1530.pth"
  dest: "d/525662143"
}
, 'DataPutReply')
Uploaded file 'model_epoch_1530.pth'
Uploading file 'model_epoch_2850.pth' to collection '09232024-FPN-dataset_v5_size-10k'...
(item {
  id: "d/525662144"
  title: "mo

KeyboardInterrupt: 